Complete training loop with best practices

Production-ready loop with validation, LR scheduling, gradient clipping, and checkpointing




In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR


# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# optional -> torch.compile() for 2-3x speed

model = torch.compile(model)

# Learning Rate Scheule: warmup + cosine annealing
total_epochs = 50
warmup_epochs = 5
warmup = LinearLR(optimizer, start_factor=0.1, total_iters = warmup_epochs)
cosine = CosineAnnealingLR(optimizer, T_max = total_epochs - warmup_epochs)
scheduler = SequentialLR(optimizer, [warmup,cosine], milestones=[warmup_epochs])


from torch.utils.data import DataLoader, TensorDataset

train_data = TensorDataset(torch.randn(640, 784), torch.randint(0, 10, (640,)))
val_data   = TensorDataset(torch.randn(160, 784), torch.randint(0, 10, (160,)))
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32)

best_val_loss = float('inf')
patience, patience_counter = 10, 0

for epoch in range(total_epochs):
  model.train()
  train_loss = 0.0
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()

    # gradient clipping before optimizer step
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    train_loss += loss.item()

  model.eval()
  val_loss, correct, total = 0.0, 0, 0 # Initialize val_loss inside the epoch loop

  with torch.no_grad():
    for inputs, targets in val_loader:
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)
      val_loss += criterion(outputs, targets).item()
      _, predicted = outputs.max(1)
      total += targets.size(0)
      correct += predicted.eq(targets).sum().item()

  avg_train = train_loss / len(train_loader)
  avg_val = val_loss / len(val_loader)
  accuracy = 100. * correct / total

  scheduler.step()

  print(f"Epoch {epoch+1}/{total_epochs} | "
    f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
    f"Acc: {accuracy:.1f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

  if avg_val < best_val_loss:
          best_val_loss = avg_val
          patience_counter = 0
          torch.save({
              'epoch': epoch,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'scheduler_state_dict': scheduler.state_dict(),
              'best_val_loss': best_val_loss,
          }, 'best_model.pt')
          print(f"  -> Saved best model (val_loss: {best_val_loss:.4f})")
  else:
          patience_counter += 1
          if patience_counter >= patience:
              print(f"Early stopping at epoch {epoch+1}")
              break

# Load best model for evaluation
checkpoint = torch.load('best_model.pt', weights_only=False)  # weights_only=False needed for optimizer/scheduler state
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

Epoch 1/50 | Train: 2.3149 | Val: 2.3079 | Acc: 8.8% | LR: 0.000084
  -> Saved best model (val_loss: 2.3079)
Epoch 2/50 | Train: 2.2985 | Val: 2.3089 | Acc: 6.9% | LR: 0.000138
Epoch 3/50 | Train: 2.2739 | Val: 2.3093 | Acc: 7.5% | LR: 0.000192
Epoch 4/50 | Train: 2.2359 | Val: 2.3103 | Acc: 10.6% | LR: 0.000246
Epoch 5/50 | Train: 2.1864 | Val: 2.3135 | Acc: 11.2% | LR: 0.000300
Epoch 6/50 | Train: 2.1072 | Val: 2.3209 | Acc: 8.1% | LR: 0.000300
Epoch 7/50 | Train: 1.9869 | Val: 2.3278 | Acc: 8.8% | LR: 0.000299
Epoch 8/50 | Train: 1.8429 | Val: 2.3481 | Acc: 10.0% | LR: 0.000297
Epoch 9/50 | Train: 1.6495 | Val: 2.3632 | Acc: 10.6% | LR: 0.000294
Epoch 10/50 | Train: 1.4354 | Val: 2.3835 | Acc: 10.6% | LR: 0.000291
Epoch 11/50 | Train: 1.1775 | Val: 2.4081 | Acc: 10.0% | LR: 0.000287
Early stopping at epoch 11
Loaded best model from epoch 1


Mixed precision training with AMP (automatic mixed precision)

Train with float16/bfloat16 for ~2x speedup and ~40% memory savings




In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# self contained and can run standalone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Linear(784,256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128,10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
num_epochs = 3

# syntheic data for demonstration
train_data = TensorDataset(torch.randn(320,784), torch.randint(0,10, (320,)))
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# GradScaler prevents gradients underflow in float16
# not needed for bfloat16 (use autocast alone)
use_amp = device.type == 'cuda'
scaler = GradScaler('cuda') if use_amp else None

# training with fp16 (requries grad scaler)

for epoch in range(num_epochs):
  model.train()
  for inputs, targets in train_loader:
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad()

    # autocast run forward pass in float16 where safe
    with autocast('cuda', enabled=use_amp):
      outputs = model(inputs)
      loss = criterion(outputs, targets)

    if use_amp:
      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
      scaler.step(optimizer)
      scaler.update()
    else:
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

# bfloat16 simpler and no scaler needed
# Preferred on Ampere+ GPUs (A100, H100, RTX 3090+)
for inputs, targets in train_loader:
  inputs = inputs.to(device)
  targets = targets.to(device)

  optimizer.zero_grad()

  # bfloat16 has sae exponent range as float32
  # no loss scaling needed
  with autocast('cuda', dtype=torch.bfloat16):
    outputs = model(inputs)
    loss = criterion(outputs, targets)

  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  optimizer.step()



# memory comparioson
def count_memory():
  allocated = torch.cuda.memory_allocated() / 1e9
  reserved = torch.cuda.memory_reserved() / 1e9
  return f"Allocated: {allocated:4f} and Reserved: {reserved:4f}"

print(f"Memory Usage: {count_memory()}")
print(f"Typical Savings: ~40-60% less memory with mixed precision")

Memory Usage: Allocated: 0.000000 and Reserved: 0.000000
Typical Savings: ~40-60% less memory with mixed precision


/tmp/ipykernel_768/767353018.py:65: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast('cuda', dtype=torch.bfloat16):


Gradient accumulation for large effective batches

Simulate batch_size=256 when GPU only fits batch_size=32